# Instruction Fine-Tuning (supervised fine-tuning - SFT) Tutorial 🚀

**Is notebook mein hum seekhenge:**
1. **Instruction Dataset Formatting**: Kaise hum dataset ko `### Instruction`, `### Input` aur `### Response` format mein align karte hain.
2. **Instruction Fine-Tuning**: Pehle se domain-fine-tuned model (`checkpoint-5`) ko use karke instruction datasets par train karna.
3. **Model Comparison**: Raw (Non-Instruction) model aur Instruction-Tuned model ke answers ko aapas mein compare karna.

In [7]:
# Sabse pehle essential libraries aur tokenizers import karte hain
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [8]:
# Base TinyLlama model ID define karte hain
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [9]:
# Tokenizer load karte hain aur verify karte hain ki padding settings correct hain
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [10]:
# Hamare pichle chapter mein trained custom domain adapter checkpoint ka path define karte hain
model_path = "./tinyllama-lora-non-instruction"

In [11]:
# Custom non-instruction adapter model load karte hain (Using bfloat16 to avoid NaN/numeric issues on MPS)
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [12]:
# Test medical prompt select karte hain evaluate karne ke liye
# Test prompt defining medical drug context
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [13]:
# Prompt tokenization aur device setting (MPS pe text push karte hain)
inputs = tokenizer(prompt, return_tensors="pt").to("mps")

In [14]:
# Generation configuration adjust karte hain warning remove karne ke liye, fir text generate karte hain
non_instruction_model.generation_config.max_length = None
# Inference execution: max_length config set to None to avoid validation warnings during generation
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [15]:
# Final text output decode aur decode display
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe was superior to Ezetimibe alone in the reduction of total cholesterol, LDL-C and apoprotein B. In addition, the combination reduced LDL-C by 35% compared with Ezetimibe alone, which is an indication of its potent ability to improve cardiovascular health.
"The approval of Atorvastatin Calcium for use in combination with Ezetimibe in China is a significant mil


## 1. Mental Health Dataset Formatting Example 📂

Hum pehle ek mental health dataset load karenge aur dekhnge ki Q&A pattern ko prompt structure mein kaise format kiya jata hai.

In [16]:
# Counseling conversation dataset ko download karte hain
from datasets import load_dataset

dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

In [17]:
# Dataset aur pehle example conversation ko print karke inspection karte hain
print(dataset)
print(" ")

print(f"Context: {dataset['Context'][0]}\n")
print(f"Response: {dataset['Response'][0]}")

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})
 
Context: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone?

Response: If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media.  Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if so

In [18]:
# Context aur Response columns ko standard Instruction Prompt format `[INST] ... [/INST]` mein transform karne ka function
def format_row(example):
    question = example['Context']
    answer = example['Response']
    example['Text'] = f"[INST] {question} [/INST] {answer}"
    return example

formatted_dataset = dataset.map(format_row)

formatted_dataset[0]['Text']

"[INST] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone? [/INST] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.B

In [19]:
# Pandas DataFrame use karke raw dataset table visualization check karte hain
import pandas as pd

df = pd.DataFrame(dataset)
df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [20]:
# Updated instructions format dataframe structure review karte hain
formatted_df = pd.DataFrame(formatted_dataset)
formatted_df.head()

,Context,Response,Text
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb...",[INST] I'm going through some things with my f...
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",[INST] I'm going through some things with my f...
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...,[INST] I'm going through some things with my f...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...,[INST] I'm going through some things with my f...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,[INST] I'm going through some things with my f...


## 2. Load Custom Pharma Instruction Dataset 💊

Hum ab custom local pharmacy/medical instruction Q&A dataset load karenge jo humne fine-tune karne ke liye write kiya hai.

In [21]:
# Local pharma instruction JSONL file load karte hain
dataset = load_dataset("json", data_files="./pharma_instruction_data.jsonl", split="train")
dataset


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [22]:
# Har entry ko `### Instruction`, `### Input` aur `### Response` format mein append karne ka function
def format_example(example):
    prompt = f"### Instruction: \n{example['instruction']}\n### Input: \n{example['input']}\n### Response: \n{example['output']}"
    return {"text": prompt}

dataset = dataset.map(format_example)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [43]:
# Formatted instruction text check karte hain jo training mein pass hoga
dataset['text'][-1]

'### Instruction: \nFrom the passage, extract two benefits and two challenges of AI in pharmaceutical R&D.\n### Input: \nArtificial intelligence (AI) is transforming pharmaceutical research by accelerating target identification, molecular docking, and compound screening. Deep learning models trained on large-scale biological datasets can predict protein–ligand binding affinities and optimize lead compounds. Integrating AI-driven insights with laboratory automation is reducing discovery timelines from years to months. However, challenges remain regarding interpretability, bias mitigation, and regulatory validation for AI-generated molecules.\n### Response: \nBenefits: (1) Faster target ID, docking, and screening; (2) Better protein–ligand affinity prediction and lead optimization. Challenges: (1) Limited interpretability and bias risks; (2) Regulatory validation of AI-generated molecules.'

In [24]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [25]:
def tokenize_fn(example):
    tokens = tokenizer(example['text'], truncation=True, padding="max_length", max_length=512)
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

tokenized = dataset.map(tokenize_fn)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [26]:
from peft import LoraConfig, get_peft_model, TaskType

## 3. LoRA Configuration & SFT (Supervised Fine-Tuning) 🛠️

Ab hum instruction dataset par model ko update karne ke liye LoRA adapters configuration setup karenge.

In [27]:
# Target adapters (q_proj, v_proj) configuration define karte hain rank r=8 ke sath
# LoRA configuration specify karte hain (Rank r=8 aur projections adapters set karte hain)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [28]:
# Custom domain model ko get_peft_model se wrap karte hain LoRA parameters insert karne ke liye
model = get_peft_model(non_instruction_model, lora_config)

/Users/kevin/Desktop/engineer/LLMs-from-scratch/.venv/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/kevin/Desktop/engineer/LLMs-from-scratch/.venv/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [29]:
# Training parameters define karte hain: 5 epochs, bf16=True device safety ke liye
args = TrainingArguments(
    output_dir="./tinyllama-lora-instruction",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [30]:
# Training instance constructor call karte hain
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized
)

In [31]:
# SFT (Instruction Fine-Tuning) training execute karne ke liye run karein
trainer.train()

/Users/kevin/Desktop/engineer/LLMs-from-scratch/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=5, training_loss=11.286538696289062, metrics={'train_runtime': 15.4581, 'train_samples_per_second': 1.617, 'train_steps_per_second': 0.323, 'total_flos': 79537058611200.0, 'train_loss': 11.286538696289062, 'epoch': 5.0})

In [32]:
# Trained weights model checkpoint aur tokenizer local store save karte hain
# Save & test the model
trainer.save_model("./tinyllama-lora-instruction")
tokenizer.save_pretrained("./tinyllama-lora-instruction")

('./tinyllama-lora-instruction/tokenizer_config.json',
 './tinyllama-lora-instruction/tokenizer.json')

## 4. Evaluation and Model Comparison 📊

Hum ab compare karenge ki non-instruction model ke comparisons mein new instruction-tuned model medical queries par kitne perfect answers deta hai.

In [33]:
# Instruction tuned weights directory path
model_path = "./tinyllama-lora-instruction"

In [34]:
# Save instruction model load karte hain bfloat16 precision support ke sath
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [35]:
# Drug question query selection
prompt = "Explain the mechanism of action of Metformin."

In [36]:
inputs = tokenizer(prompt, return_tensors="pt").to("mps")

In [37]:
# Generation configs setting clean warning, and run generation
instruction_model.generation_config.max_length = None
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [38]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain the mechanism of action of Metformin. Explain the mechanism of action of Metformin.
Explain the mechanism of action of Metformin.
Metformin is a member of the insulin sensitizer family. The insulin-sensitizers have the ability to increase glucose utilization by decreasing the expression of insulin receptors in tissues and cells. This increases the glucose uptake in these tissues/cells through a mechanism which is unknown.



In [39]:
# Generation configs setting clean warning, and run generation
instruction_model.generation_config.max_length = None
prompt = "### Instruction:\nWhat is Ezetimibe?\n### Input:\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("mps")

outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

### Instruction:
What is Ezetimibe?
### Input:

### Response:

### Sample Test Case 1:
```
3
6
10
```


### Expected Output:
Ezetimibe=2.5


### Expected Output:
Ezetimibe=5



In [40]:
# Multiple medical questions standard benchmark array define karte hain
questions = [
    "Explain the mechanism of action of Metformin.",
    "List two advantages of combining Atorvastatin with Ezetimibe.",
    "Summarize how mRNA vaccines work and mention one current research focus."
]

In [41]:
# Tokenize prompt and push tensors to MPS device
for q in questions:
    print("Question:", q)
    print("\n--- Non-instruction model ---")
    inputs = tokenizer(q, return_tensors="pt").to("mps")
    outputs = non_instruction_model.generate(**inputs, max_new_tokens=80)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("mps")
    outputs = instruction_model.generate(**inputs, max_new_tokens=100)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("="*80, "\n")

Question: Explain the mechanism of action of Metformin.

--- Non-instruction model ---
Explain the mechanism of action of Metformin.
Explain the mechanism of action of Metformin.
Metformin is a drug that is used to treat diabetes. It is a type of drug that is used to treat diabetes. It is a type of drug that is used to treat diabetes. It is a type of drug that is used to treat diabetes. It is a type of drug that

--- Instruction-tuned model ---
### Instruction:
Explain the mechanism of action of Metformin.
### Input:

### Response:

### Expected Output:

### Test Cases:

### Test Cases:

### Test Cases:

### Test Cases:

### Test Cases:

### Test Cases:

### Test Cases:

### Test Cases:

### Test Cases:

### Test Cases:

### Test Cases:

### Test

Question: List two advantages of combining Atorvastatin with Ezetimibe.

--- Non-instruction model ---
List two advantages of combining Atorvastatin with Ezetimibe.
Answer: Atorvastatin is a statin drug that is used to lower cholesterol. Ezet